# 01 - Data Audit, Schema Verification & Data Cleaning
**Project:** Strategic HRM and Corporate Layoffs: Patterns, Drivers, Workforce Risk and Organizational Restructuring  
**Domain:** Strategic Human Resource Management & Organizational Behaviour  

---

### Objectives
1. Ingest raw event-level tracking data directly from Layoffs.fyi / Kaggle (`swaptr/layoffs-2022`).
2. Audit exact column structures, data types, and boundary constraints.
3. Conduct exhaustive missing-data analysis (distinguishing joint vs. isolated non-reporting).
4. Disambiguate duplicate dispatches (identical scrapes vs. multi-site corporate restructuring).
5. Clean and standardize all variables into a verified analysis dataset.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add project root to path
sys.path.append('..')
from src.config import RAW_DATA_PATH, CLEANED_EVENTS_PATH
from src.cleaning import load_raw_data, audit_raw_data, clean_dataset

# 1. Ingest Raw Dataset
df_raw = load_raw_data(RAW_DATA_PATH)
print(f"Raw Dataset Shape: {df_raw.shape[0]:,} rows by {df_raw.shape[1]} columns")
df_raw.head()


### 2. Comprehensive Pre-Cleaning Audit
We inspect non-reporting rates across variables. In corporate downsizing datasets, missingness is not random: large enterprises report headcounts, while venture-backed startups often report percentages.


In [ ]:
audit = audit_raw_data(df_raw)
print("=== Column Missingness Audit ===")
for col, pct in audit["missing_percentages"].items():
    print(f"  {col:20s}: {audit['missing_counts'][col]:5d} missing ({pct:5.1f}%)")

print("
=== Joint Severity Metric Configuration ===")
print(f"  Both Headcount & Percentage Present : {audit['both_severity_present']:,}")
print(f"  At Least One Metric Present         : {audit['at_least_one_severity_present']:,}")
print(f"  Both Metrics Missing                : {audit['both_severity_missing']:,}")


### 3. Cleaning & Standardization Pipeline
We normalize text fields, parse ISO dates, validate numeric constraints ($0.0 \le 	ext{percentage} \le 1.0$), and deduplicate redundant scrapes.


In [ ]:
df_clean, meta = clean_dataset(df_raw)
print(f"Initial Records : {meta['initial_rows']:,}")
print(f"Exact Duplicates Removed : {meta['removed_exact_event_duplicates']:,}")
print(f"Cleaned Records Retained : {meta['final_rows']:,}")
print(f"Temporal Span   : {meta['min_date']} to {meta['max_date']}")
print(f"Unique Companies: {meta['unique_companies']:,}")


### Strategic HRM Takeaway
Data cleaning reveals that over 30% of tech downsizing records lack either headcount or percentage. Consequently, research that relies solely on one metric introduces severe selection bias. Our analytical strategy explicitly tracks both magnitude and intensity.
